In [1]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

In [2]:
from dotenv import load_dotenv

load_dotenv("../environments/.env")

True

In [3]:
query_fact = {
    "filter": {
        "property": "title",
        "rich_text": {"contains": "29c4eaab90d281ce8429f5961c66ba77"},
    }
}

In [4]:
query_talhoes = {
    "filter": {
        "property": "farm",
        "relation": {"contains": "29b4eaab-90d2-80f7-8718-e9ace813e33c"},
    }
}

In [5]:
import os
from notion_client import Client
from uuid import UUID

fact_id = UUID("29c4eaab90d28015a00fe7e0faaa5c22")
talhoes_id = UUID("29b4eaab90d280d594e9d765abf12e59")

with Client(auth=os.environ["NOTION_TOKEN"]) as notion:
    fact_db = notion.databases.retrieve(fact_id)
    fact_data_source_id = fact_db["data_sources"][0]["id"]
    print(fact_db)
    print(fact_data_source_id)

    talhoes_db = notion.databases.retrieve(talhoes_id)
    talhoes_data_source_id = talhoes_db["data_sources"][0]["id"]
    print(talhoes_db)
    print(talhoes_data_source_id)

    fact_data_source = notion.data_sources.query(fact_data_source_id, **query_fact)
    print(fact_data_source)

    talhoes_data_source = notion.data_sources.query(
        talhoes_data_source_id, **query_talhoes
    )
    print(talhoes_data_source)

{'object': 'database', 'id': '29c4eaab-90d2-8015-a00f-e7e0faaa5c22', 'title': [{'type': 'text', 'text': {'content': 'Fato', 'link': None}, 'annotations': {'bold': False, 'italic': False, 'strikethrough': False, 'underline': False, 'code': False, 'color': 'default'}, 'plain_text': 'Fato', 'href': None}], 'description': [], 'parent': {'type': 'workspace', 'workspace': True}, 'is_inline': False, 'in_trash': False, 'is_locked': False, 'created_time': '2025-10-30T00:23:39.788+00:00', 'last_edited_time': '2025-10-31T00:58:53.445+00:00', 'data_sources': [{'id': '29c4eaab-90d2-8089-bf7a-000b77d60257', 'name': 'Fato'}], 'icon': None, 'cover': None, 'url': 'https://www.notion.so/29c4eaab90d28015a00fe7e0faaa5c22', 'public_url': None, 'request_id': 'f85c280b-e2a5-4606-9611-8f6849d603e2'}
29c4eaab-90d2-8089-bf7a-000b77d60257
{'object': 'database', 'id': '29b4eaab-90d2-80d5-94e9-d765abf12e59', 'title': [{'type': 'text', 'text': {'content': 'Talhoes', 'link': None}, 'annotations': {'bold': False, 'it

In [6]:
from fastapi import HTTPException
from typing import Any, Dict
from services.notion.notion_service import NotionService


def gerar_relatorio(template: str, page_id: str) -> Dict[str, Any]:
    try:
        notion = NotionService()
        fact_ds_id, talhoes_ds_id = notion.get_fact_and_talhoes_data_sources()

        # 1) Query no data source do FACT pelo page_id
        query_fact = {
            "filter": {"property": "title", "rich_text": {"contains": page_id}}
        }
        fact_ds_result = notion.query_data_source(fact_ds_id, query_fact)
        fact_items = notion.parse_data_source_results(fact_ds_result)

        if not fact_items:
            raise HTTPException(
                status_code=404, detail=f"FACT page {page_id} não encontrada"
            )

        # 2) Extrair o campo 'farm' do primeiro item retornado
        fact_item = fact_items[0]
        farm_ids = fact_item.get("farm", [])

        if not isinstance(farm_ids, list):
            farm_ids = [farm_ids] if farm_ids else []

        # 3) Query no data source de talhões usando os farm_ids
        talhoes_items = []
        if farm_ids:
            # Para cada farm_id, fazer query no data source de talhões
            for farm_id in farm_ids:
                query_talhoes = {
                    "filter": {"property": "farm", "relation": {"contains": farm_id}}
                }
                talhoes_ds_result = notion.query_data_source(
                    talhoes_ds_id, query_talhoes
                )
                talhoes_parcial = notion.parse_data_source_results(talhoes_ds_result)
                talhoes_items.extend(talhoes_parcial)

        # 4) Agregar informações úteis (quantidade, nomes, áreas, etc.)
        resumo_talhoes = notion.summarize_talhoes(
            talhoes_items, area_property_name="area"
        )

        # 5) Tentar descobrir o nome da fazenda
        farm_name = None
        if farm_ids:
            farm_name = notion.resolve_page_title(farm_ids[0])

        return {
            "template": template,
            "page_id": page_id.replace("-", ""),
            "fact": {
                "total_itens": len(fact_items),
                "itens": fact_items,
            },
            "talhoes": {
                "ids": farm_ids,
                "itens": talhoes_items,
                "resumo": resumo_talhoes,
                "fazenda_nome": farm_name,
            },
        }
    except Exception as exc:
        if isinstance(exc, HTTPException):
            raise exc
        raise HTTPException(status_code=500, detail=str(exc))

In [7]:
gerar_relatorio("relatorio_padrao", "29c4eaab90d281ce8429f5961c66ba77")

{'template': 'relatorio_padrao',
 'page_id': '29c4eaab90d281ce8429f5961c66ba77',
 'fact': {'total_itens': 1,
  'itens': [{'nome_fazenda': {'type': 'array',
     'array': [{'type': 'rich_text',
       'rich_text': [{'type': 'text',
         'text': {'content': 'Fazenda Riviera', 'link': None},
         'annotations': {'bold': False,
          'italic': False,
          'strikethrough': False,
          'underline': False,
          'code': False,
          'color': 'default'},
         'plain_text': 'Fazenda Riviera',
         'href': None}]}],
     'function': 'show_original'},
    'farm': ['29b4eaab90d280f78718e9ace813e33c'],
    'page_id': '29c4eaab90d281ce8429f5961c66ba77',
    '_id': '29c4eaab90d281d789afcb9117143c9b',
    '_url': 'https://www.notion.so/29c4eaab90d281ce8429f5961c66ba77-29c4eaab90d281d789afcb9117143c9b',
    '_title': '29c4eaab90d281ce8429f5961c66ba77'}]},
 'talhoes': {'ids': ['29b4eaab90d280f78718e9ace813e33c'],
  'itens': [{'nome_fazenda': {'type': 'array',
     '

In [9]:
from services.relatorio_service import RelatorioService

# Criar instância do serviço
relatorio_service = RelatorioService()

# Gerar relatório
resultado = relatorio_service.gerar_relatorio(
    template="relatorio_padrao", page_id="29c4eaab90d281ce8429f5961c66ba77"
)

resultado

{'template': 'relatorio_padrao',
 'page_id': '29c4eaab90d281ce8429f5961c66ba77',
 'fact': {'total_itens': 1,
  'itens': [{'nome_fazenda': {'type': 'array',
     'array': [{'type': 'rich_text',
       'rich_text': [{'type': 'text',
         'text': {'content': 'Fazenda Riviera', 'link': None},
         'annotations': {'bold': False,
          'italic': False,
          'strikethrough': False,
          'underline': False,
          'code': False,
          'color': 'default'},
         'plain_text': 'Fazenda Riviera',
         'href': None}]}],
     'function': 'show_original'},
    'farm': ['29b4eaab90d280f78718e9ace813e33c'],
    'page_id': '29c4eaab90d281ce8429f5961c66ba77',
    '_id': '29c4eaab90d281d789afcb9117143c9b',
    '_url': 'https://www.notion.so/29c4eaab90d281ce8429f5961c66ba77-29c4eaab90d281d789afcb9117143c9b',
    '_title': '29c4eaab90d281ce8429f5961c66ba77'}]},
 'talhoes': {'farm_ids': ['29b4eaab90d280f78718e9ace813e33c'],
  'itens': [{'nome_fazenda': {'type': 'array',
 